In [1]:
import abc
import ast
import random
import re
import ollama
from typing import Dict, List, Tuple, Any
import numpy as np
import gymnasium as gym
from gymnasium.spaces import Discrete, Box
from pettingzoo import ParallelEnv

In [2]:
# ==========================================
# 1. ABSTRACT POSG & WORLD MODEL INTERFACE
# ==========================================

class AbstractPOSG(abc.ABC):
    """
    Core interface for Partially Observable Stochastic Games (POSGs).
    """
    @property
    @abc.abstractmethod
    def players(self) -> List[str]:
        """List of player IDs (e.g. ['player_1', 'player_2'])."""
        pass

    @abc.abstractmethod
    def legal_actions(self, state: Any, player: str) -> List[str]:
        """Returns legal actions for a given player in a given state."""
        pass

    @abc.abstractmethod
    def transition(self, state: Any, joint_action: Dict[str, str]) -> Dict[Any, float]:
        """
        Returns a distribution over next states given current state and joint action.
        e.g., {next_state_1: 0.8, next_state_2: 0.2}
        """
        pass

    @abc.abstractmethod
    def observation(self, state: Any, joint_action: Dict[str, str], next_state: Any, player: str) -> str:
        """Returns the local observation for a player."""
        pass

    @abc.abstractmethod
    def reward(self, state: Any, joint_action: Dict[str, str], next_state: Any, player: str) -> float:
        """Returns the immediate reward for a player."""
        pass

    @abc.abstractmethod
    def is_terminal(self, state: Any) -> bool:
        """Returns True if the state is terminal."""
        pass

    @abc.abstractmethod
    def initial_state(self) -> Any:
        """Returns the initial state of the game."""
        pass


class WorldModelInspector:
    """Helper to inspect state transitions, transition probability distributions, and observations."""
    def __init__(self, posg: AbstractPOSG):
        self.posg = posg

    def inspect_transition(self, state: Any, joint_action: Dict[str, str]):
        print(f"\n--- World Model Inspection ---")
        print(f"Current State: {state}")
        print(f"Joint Action:  {joint_action}")
        
        dist = self.posg.transition(state, joint_action)
        print("Transition Distribution:")
        for n_state, prob in dist.items():
            print(f"  └─> Next State: '{n_state}' (Prob: {prob:.2f})")
            for p in self.posg.players:
                obs = self.posg.observation(state, joint_action, n_state, p)
                rew = self.posg.reward(state, joint_action, n_state, p)
                print(f"      [{p}] Obs: '{obs}' | Reward: {rew}")


In [3]:
# ==========================================
# 2. STRICT POSG VALIDATOR
# ==========================================

def validate_posg(posg_cls) -> Tuple[bool, List[str]]:
    """
    Validates that a generated class strictly follows AbstractPOSG contract and behavior rules.
    """
    logs = []
    
    # 1. Instance check
    try:
        posg = posg_cls()
    except Exception as e:
        return False, [f"Instantiation failed: {e}"]

    if not isinstance(posg, AbstractPOSG):
        logs.append("ERROR: POSG class does not inherit from AbstractPOSG.")
        return False, logs

    # 2. Players check
    players = posg.players
    if not isinstance(players, list) or len(players) < 2:
        logs.append(f"ERROR: Invalid players list: {players}. Needs at least 2 players.")
        return False, logs

    # 3. Initial state
    state = posg.initial_state()
    if state is None:
        logs.append("ERROR: initial_state() returned None.")
        return False, logs

    # 4. Actions & Transitions verification
    for p in players:
        actions = posg.legal_actions(state, p)
        if not actions or not isinstance(actions, list):
            logs.append(f"ERROR: No legal actions returned for player {p}.")
            return False, logs

    # Test joint action
    sample_joint_action = {p: posg.legal_actions(state, p)[0] for p in players}
    
    # Check legal action restriction
    for p, act in sample_joint_action.items():
        if act not in posg.legal_actions(state, p):
            logs.append(f"ERROR: Action '{act}' not declared in legal_actions for {p}.")

    # Check transition distribution
    dist = posg.transition(state, sample_joint_action)
    if not isinstance(dist, dict) or len(dist) == 0:
        logs.append("ERROR: transition() must return a non-empty dict mapping state -> probability.")
        return False, logs

    prob_sum = sum(dist.values())
    if not np.isclose(prob_sum, 1.0):
        logs.append(f"ERROR: Transition probabilities do not sum to 1.0 (Sum: {prob_sum}).")
        return False, logs

    # Check observations & rewards across distribution states
    for next_state in dist.keys():
        for p in players:
            try:
                obs = posg.observation(state, sample_joint_action, next_state, p)
                rew = posg.reward(state, sample_joint_action, next_state, p)
                if not isinstance(obs, str):
                    logs.append(f"ERROR: Observation for {p} must be string. Got {type(obs)}.")
                if not isinstance(rew, (int, float)):
                    logs.append(f"ERROR: Reward for {p} must be numeric. Got {type(rew)}.")
            except Exception as e:
                logs.append(f"ERROR during obs/reward evaluation: {e}")

    # Check terminal state behavior
    is_term = posg.is_terminal(state)
    if not isinstance(is_term, bool):
        logs.append("ERROR: is_terminal() must return a boolean.")

    logs.append("✅ AbstractPOSG Validation Passed!")
    return True, logs

In [4]:
# ==========================================
# 3. PETTINGZOO PARALLEL ENV WRAPPER
# ==========================================

class POSGToPettingZoo(ParallelEnv):
    """Automatically wraps any validated AbstractPOSG into a standard PettingZoo ParallelEnv."""
    
    metadata = {"name": "posg_pettingzoo_v0"}

    def __init__(self, posg: AbstractPOSG):
        super().__init__()
        self.posg = posg
        self.possible_agents = posg.players[:]
        self.agents = self.possible_agents[:]
        
        # Determine unique actions per agent
        init_st = posg.initial_state()
        self._action_maps = {}
        self._rev_action_maps = {}
        self.action_spaces = {}
        self.observation_spaces = {}

        for p in self.possible_agents:
            acts = posg.legal_actions(init_st, p)
            self._action_maps[p] = {i: act for i, act in enumerate(acts)}
            self._rev_action_maps[p] = {act: i for i, act in enumerate(acts)}
            self.action_spaces[p] = Discrete(len(acts))
            # Categorical string observations represented as text string inside info/obs
            self.observation_spaces[p] = gym.spaces.Text(max_length=512)

        self.current_state = None

    def reset(self, seed=None, options=None):
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        self.agents = self.possible_agents[:]
        self.current_state = self.posg.initial_state()
        
        # Initial empty observation step
        obs = {p: "GAME_START" for p in self.agents}
        infos = {p: {} for p in self.agents}
        return obs, infos

    def step(self, actions: Dict[str, int]):
        if not self.agents:
            return {}, {}, {}, {}, {}

        # Convert discrete action indices to string action names
        str_actions = {
            p: self._action_maps[p][act_idx] 
            for p, act_idx in actions.items() 
            if p in self.agents
        }

        # Query transition distribution & sample next state
        dist = self.posg.transition(self.current_state, str_actions)
        next_states = list(dist.keys())
        probs = list(dist.values())
        sampled_next_state = random.choices(next_states, weights=probs, k=1)[0]

        # Observations and Rewards
        obs = {}
        rewards = {}
        for p in self.agents:
            obs[p] = self.posg.observation(self.current_state, str_actions, sampled_next_state, p)
            rewards[p] = float(self.posg.reward(self.current_state, str_actions, sampled_next_state, p))

        # Update environment state
        self.current_state = sampled_next_state
        terminated_flag = self.posg.is_terminal(self.current_state)

        terminations = {p: terminated_flag for p in self.agents}
        truncations = {p: False for p in self.agents}
        infos = {p: {} for p in self.agents}

        if terminated_flag:
            self.agents = []

        return obs, rewards, terminations, truncations, infos

In [8]:
# ==========================================
# CELL 4: PROMPT & PYTHON CODE GENERATION ENGINE
# ==========================================

PROMPT_TEMPLATE = """
You are a world-model builder. You turn a natural-language SITUATION into a formal
stochastic-game world model implemented as a single Python class.

The Python class will be executed by a separate evaluation framework and wrapped as a
PettingZoo ParallelEnv.

Your job is to represent the dynamics of the situation:
- who acts
- what actions they can take
- what states can occur
- what observations players receive
- what follows from each joint action

Do not invent unnecessary details.

If the situation is under-specified, choose one small, concrete instance and commit to it.

# Required Python Interface

You MUST define exactly one main class:

class GeneratedPOSG(AbstractPOSG):

The class MUST implement:

    @property
    def players(self) -> List[str]: ...

    def legal_actions(self, state, player) -> List[str]: ...

    def transition(self, state, joint_action) -> Dict[Any, float]: ...

    def observation(self, state, joint_action, next_state, player) -> str: ...

    def reward(self, state, joint_action, next_state, player) -> float: ...

    def is_terminal(self, state) -> bool: ...

    def initial_state(self) -> Any: ...

# Method requirements

## players

Return a list of player IDs.

Example:

["player_1", "player_2"]

Do not create a separate nature, environment, or chance player.

## initial_state

Return the initial state.

Prefer simple immutable states such as strings:

"s0"

## legal_actions

Return the actions available to the specified player in the specified state.

Example:

def legal_actions(self, state, player):
    if state == "s0":
        return ["rock", "paper", "scissors"]
    return []

## transition

The method receives:

transition(self, state, joint_action)

where joint_action is a dictionary such as:

{{
    "player_1": "rock",
    "player_2": "scissors"
}}

Return a dictionary mapping next states to probabilities:

{{
    "player_1_wins": 1.0
}}

For stochastic outcomes:

{{
    "success": 0.7,
    "failure": 0.3
}}

Probabilities MUST be non-negative and sum to exactly 1.0.

Do NOT sample randomness inside transition.

## observation

Return what the specified player observes after the transition.

Do not reveal information that the player should not know.

## reward

Return a numeric reward for the specified player.

If the situation does not explicitly specify a reward or payoff, return:

0.0

Do not invent a reward function merely because the situation is competitive.

## is_terminal

Return True for terminal states and False otherwise.

# Simultaneous actions

If multiple players act simultaneously, joint_action must contain an action for every
acting player.

Do not convert simultaneous decisions into sequential decisions.

# Stochasticity

Represent chance through the probability distribution returned by transition.

Do not create a separate chance player.

# State representation

Keep states minimal.

A state should contain enough information to determine:
- available actions
- observations
- transitions
- whether the state is terminal

Do not create unnecessary narrative states.

# Metadata

Where useful, include simple declarative metadata:

player_descriptions = {{
    "player_1": "...",
    "player_2": "..."
}}

state_variables = {{
    "variable": {{
        "type": "str",
        "description": "..."
    }}
}}

action_catalogue = {{
    "action": "description"
}}

Keep metadata simple.

# PettingZoo compatibility

The implementation will later be wrapped automatically in a PettingZoo ParallelEnv.

Therefore:
- actions must be discrete
- simultaneous players must be supported
- terminal states must be detectable
- reset must return the initial state
- transitions must be executable

Do NOT implement PettingZoo yourself.

# Output requirements

Return ONLY executable Python code.

The output MUST:

1. Define GeneratedPOSG.
2. Inherit from AbstractPOSG.
3. Implement all required methods.
4. Contain no markdown fences.
5. Contain no explanations.
6. Use only the standard library and the already-provided AbstractPOSG interface.
7. Be directly executable using exec().

# Situation

{game_description}
"""


def generate_posg_code(
    game_description: str,
    model_name: str = "qwen2.5-coder:latest"
) -> str:

    prompt = PROMPT_TEMPLATE.format(
        game_description=game_description
    )

    print(f"Using LLM: {model_name}")
    print("Generating Python POSG implementation...")

    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.0
        }
    )

    content = response["message"]["content"].strip()

    # Handle models that still return markdown fences despite instructions
    match = re.search(
        r"```python\s*(.*?)\s*```",
        content,
        re.DOTALL
    )

    if match:
        content = match.group(1).strip()

    return content

In [11]:
# ==========================================
# CELL 5: LLM → PYTHON POSG → VALIDATION → PETTINGZOO
# ==========================================

import typing

def run_posg_experiment(
    game_description: str,
    model_name: str = "qwen2.5-coder:latest"
):

    print("=" * 70)
    print("PYTHON POSG WORLD-MODEL EXPERIMENT")
    print("=" * 70)

    # --------------------------------------------------
    # STEP 1: Natural-language situation
    # --------------------------------------------------

    print("\nSTEP 1: NATURAL-LANGUAGE SITUATION")
    print("-" * 70)
    print(game_description)

    # --------------------------------------------------
    # STEP 2: Ask the LLM to generate the world model
    # --------------------------------------------------

    print("\nSTEP 2: LLM WORLD-MODEL GENERATION")
    print("-" * 70)

    raw_code = generate_posg_code(
        game_description,
        model_name=model_name
    )

    print("\nGenerated Python world model:")
    print("=" * 70)
    print(raw_code)
    print("=" * 70)

    # --------------------------------------------------
    # STEP 3: Execute generated Python
    # --------------------------------------------------

    print("\nSTEP 3: EXECUTING GENERATED WORLD MODEL")
    print("-" * 70)

    namespace = {
        "AbstractPOSG": AbstractPOSG,
        "List": List,
        "Dict": Dict,
        "Tuple": Tuple,
        "Any": Any,
        "Optional": typing.Optional,  # <--- FIXED: avoids NameError if Optional wasn't imported
        "abc": abc
    }

    try:
        exec(raw_code, namespace)
        print("PASS: Generated Python code executed successfully.")
    except Exception as e:
        print("FAIL: Generated code could not be executed.")
        print(f"Error: {type(e).__name__}: {e}")
        return None

    # --------------------------------------------------
    # STEP 4: Check that GeneratedPOSG exists
    # --------------------------------------------------

    print("\nSTEP 4: CHECKING GENERATEDPOSG")
    print("-" * 70)

    posg_cls = namespace.get("GeneratedPOSG")

    if posg_cls is None:
        print("FAIL: GeneratedPOSG class was not found.")
        return None

    print("PASS: GeneratedPOSG class found.")

    # --------------------------------------------------
    # STEP 5: Validate AbstractPOSG contract
    # --------------------------------------------------

    print("\nSTEP 5: ABSTRACTPOSG CONTRACT VALIDATION")
    print("-" * 70)

    valid, logs = validate_posg(posg_cls)

    for log_msg in logs:
        print(log_msg)

    if not valid:
        print("\nFAIL: AbstractPOSG validation failed.")
        return None

    print("\nPASS: AbstractPOSG validation passed.")

    # --------------------------------------------------
    # STEP 6: Instantiate world model
    # --------------------------------------------------

    print("\nSTEP 6: WORLD MODEL")
    print("-" * 70)

    posg_instance = posg_cls()

    print("World model object:")
    print(posg_instance)

    print("\nPlayers:")
    print(posg_instance.players)

    print("\nInitial state:")
    initial_state = posg_instance.initial_state()
    print(initial_state)

    # --------------------------------------------------
    # STEP 7: Inspect legal actions
    # --------------------------------------------------

    print("\nSTEP 7: LEGAL ACTIONS")
    print("-" * 70)

    for player in posg_instance.players:
        actions = posg_instance.legal_actions(
            initial_state,
            player
        )
        print(f"{player}: {actions}")

    # --------------------------------------------------
    # STEP 8: Construct a sample joint action
    # --------------------------------------------------

    print("\nSTEP 8: SAMPLE JOINT ACTION")
    print("-" * 70)

    joint_action = {}

    for player in posg_instance.players:
        actions = posg_instance.legal_actions(
            initial_state,
            player
        )
        if actions:
            joint_action[player] = actions[0]

    print("Joint action:")
    print(joint_action)

    # --------------------------------------------------
    # STEP 9: Query the world-model transition
    # --------------------------------------------------

    print("\nSTEP 9: WORLD-MODEL TRANSITION")
    print("-" * 70)

    try:
        transition = posg_instance.transition(
            initial_state,
            joint_action
        )

        print("Transition distribution:")
        print(transition)

        probability_sum = sum(transition.values())

        print(f"\nProbability sum: {probability_sum}")

        if abs(probability_sum - 1.0) < 1e-6:
            print("PASS: Transition probabilities sum to 1.0.")
        else:
            print("FAIL: Transition probabilities do not sum to 1.0.")

    except Exception as e:
        print("FAIL: Transition execution failed.")
        print(f"{type(e).__name__}: {e}")
        return None

    # --------------------------------------------------
    # STEP 10: Inspect resulting state
    # --------------------------------------------------

    print("\nSTEP 10: RESULTING WORLD STATE")
    print("-" * 70)

    next_state = max(
        transition,
        key=transition.get
    )

    print("Most likely next state:")
    print(next_state)

    print("\nTerminal?")
    print(posg_instance.is_terminal(next_state))

    # --------------------------------------------------
    # STEP 11: Inspect observations and rewards
    # --------------------------------------------------

    print("\nSTEP 11: OBSERVATIONS AND REWARDS")
    print("-" * 70)

    observations = {}
    rewards = {}

    for player in posg_instance.players:
        observations[player] = posg_instance.observation(
            initial_state,
            joint_action,
            next_state,
            player
        )

        rewards[player] = posg_instance.reward(
            initial_state,
            joint_action,
            next_state,
            player
        )

    print("Observations:")
    print(observations)

    print("\nRewards:")
    print(rewards)

    # --------------------------------------------------
    # STEP 12: PettingZoo wrapper
    # --------------------------------------------------

    print("\nSTEP 12: PETTINGZOO PARALLEL ENVIRONMENT")
    print("-" * 70)

    try:
        pz_env = POSGToPettingZoo(
            posg_instance
        )
        print("PASS: POSG successfully wrapped as PettingZoo ParallelEnv.")
    except Exception as e:
        print("FAIL: PettingZoo wrapper failed.")
        print(f"{type(e).__name__}: {e}")
        return None

    # --------------------------------------------------
    # STEP 13: PettingZoo reset
    # --------------------------------------------------

    print("\nSTEP 13: PETTINGZOO RESET")
    print("-" * 70)

    try:
        obs, infos = pz_env.reset(
            seed=42
        )

        print("Initial observations:")
        print(obs)

        print("Agents:")
        print(pz_env.agents)

        print("PASS: PettingZoo reset successful.")

    except Exception as e:
        print("FAIL: PettingZoo reset failed.")
        print(f"{type(e).__name__}: {e}")
        return None

    # --------------------------------------------------
    # STEP 14: PettingZoo action sampling
    # --------------------------------------------------

    print("\nSTEP 14: PETTINGZOO ACTION SAMPLING")
    print("-" * 70)

    try:
        actions = {
            agent: pz_env.action_spaces[agent].sample()
            for agent in pz_env.agents
        }

        print("Sampled actions:")
        print(actions)

    except Exception as e:
        print("FAIL: Could not sample actions.")
        print(f"{type(e).__name__}: {e}")
        return None

    # --------------------------------------------------
    # STEP 15: PettingZoo environment step
    # --------------------------------------------------

    print("\nSTEP 15: PETTINGZOO STEP")
    print("-" * 70)

    try:
        (
            next_obs,
            rewards,
            terminations,
            truncations,
            infos
        ) = pz_env.step(actions)

        print("Next observations:")
        print(next_obs)

        print("\nRewards:")
        print(rewards)

        print("\nTerminations:")
        print(terminations)

        print("\nTruncations:")
        print(truncations)

        print("\nPASS: PettingZoo environment step successful.")

    except Exception as e:
        print("FAIL: PettingZoo step failed.")
        print(f"{type(e).__name__}: {e}")
        return None

    # --------------------------------------------------
    # FINAL RESULT
    # --------------------------------------------------

    print("\n" + "=" * 70)
    print("EXPERIMENT COMPLETE")
    print("=" * 70)

    return {
        "generated_code": raw_code,
        "world_model": posg_instance,
        "transition": transition,
        "next_state": next_state,
        "observations": observations,
        "rewards": rewards,
        "pettingzoo_env": pz_env
    }

In [12]:
# ==========================================
# CELL 6: RUN G.1 ROCK-PAPER-SCISSORS
# ==========================================

g1_rps_description = """
Two people sit down to play a single round of rock-paper-scissors.

Both players simultaneously choose one of:
rock, paper, or scissors.

Rock beats scissors.
Scissors beats paper.
Paper beats rock.
Identical choices result in a draw.

The round ends immediately after both players reveal their choices.
"""

experiment_result = run_posg_experiment(
    game_description=g1_rps_description,
    model_name="qwen2.5-coder:latest"
)

PYTHON POSG WORLD-MODEL EXPERIMENT

STEP 1: NATURAL-LANGUAGE SITUATION
----------------------------------------------------------------------

Two people sit down to play a single round of rock-paper-scissors.

Both players simultaneously choose one of:
rock, paper, or scissors.

Rock beats scissors.
Scissors beats paper.
Paper beats rock.
Identical choices result in a draw.

The round ends immediately after both players reveal their choices.


STEP 2: LLM WORLD-MODEL GENERATION
----------------------------------------------------------------------
Using LLM: qwen2.5-coder:latest
Generating Python POSG implementation...

Generated Python world model:
class GeneratedPOSG(AbstractPOSG):

    @property
    def players(self) -> List[str]:
        return ["player_1", "player_2"]

    def legal_actions(self, state, player) -> List[str]:
        if state == "s0":
            return ["rock", "paper", "scissors"]
        return []

    def transition(self, state, joint_action) -> Dict[Any, floa

In [16]:
!pip install dill

In [17]:
import dill

# Save the world model instance directly
with open("world_model.pth", "wb") as f:
    dill.dump(experiment_result["world_model"], f)

print("Saved World Model to local file: world_model.pth")

Saved World Model to local file: world_model.pth


In [18]:
import dill

# Reload the model back into memory
with open("world_model.pth", "rb") as f:
    loaded_wm = dill.load(f)

# Verify it works
print("Reloaded World Model Players:", loaded_wm.players)
print("Reloaded Initial State:", loaded_wm.initial_state())

Reloaded World Model Players: ['player_1', 'player_2']
Reloaded Initial State: s0
